# Создание пайплайна

In [9]:
def pipeline():
    steps = []

    def step(func):
        steps.append(func.__name__)
        
        def wrapped():
            print(f"Running step: {func.__name__}")
            func()
        
    
        step.get_all = lambda: steps
        
        return wrapped
    
    return step

In [10]:
step = pipeline()

@step
def collect_data():
    print("collect data")

@step
def preprocess_data():
    print("preprocess data")

print(step.get_all())  

['collect_data', 'preprocess_data']


# Спецификация зависимостей

In [4]:
def pipeline():
    steps = {}
    dependencies = {}

    def step(func=None, depends_on=None):
        nonlocal steps, dependencies
        depends_on = depends_on or []
        
        def decorator(f):
            steps[f.__name__] = f
            dependencies[f.__name__] = depends_on

            def wrapped():
                print(f"Running step: {f.__name__}")
                f()
            
            wrapped.get_dependencies = lambda: dependencies.get(f.__name__, [])
            
            return wrapped
        
        if func:
            return decorator(func)
        else:
            return decorator

    return step

In [5]:
step = pipeline()

@step
def collect_data():
    print("collect data")

@step(depends_on=["collect_data"])
def preprocess_data():
    print("preprocess data")

print(collect_data.get_dependencies())      
print(preprocess_data.get_dependencies())   


[]
['collect_data']


# Визуализация пайплайна

In [6]:
from graphviz import Digraph

def pipeline():
    steps = {}
    dependencies = {}

    def step(func=None, depends_on=None):
        nonlocal steps, dependencies
        depends_on = depends_on or []

        def decorator(f):
            steps[f.__name__] = f
            dependencies[f.__name__] = depends_on

            def wrapped():
                print(f"Running step: {f.__name__}")
                f()
            
            wrapped.get_dependencies = lambda: dependencies.get(f.__name__, [])
            
            return wrapped

        if func:
            return decorator(func)
        else:
            return decorator

    def graph():
        dot = Digraph(comment='Pipeline Dependency Graph')
        for step_name in steps:
            dot.node(step_name, step_name)
        for step_name, deps in dependencies.items():
            for dep in deps:
                dot.edge(dep, step_name)
        return dot
    
    step.graph = graph
    return step


# Запуск зависимостей

In [7]:
step = pipeline()

@step
def collect_data():
    print("collect data")

@step(depends_on=["collect_data"])
def preprocess_data():
    print("preprocess data")

@step(depends_on=["collect_data"])
def modify_data():
    print("modify data")

@step(depends_on=["modify_data"])
def extract_features():
    print("extract features")

@step(depends_on=["extract_features"])
def filter_features():
    print("filter features")

@step(depends_on=["preprocess_data", "modify_data"])
def merge_data():
    print("merge data")

@step(depends_on=["merge_data", "filter_features"])
def handle_data():
    print("handle data")

pipeline_graph = step.graph()
pipeline_graph.render("pipeline_graph", view=True)

'pipeline_graph.pdf'